# Unit Conversion

The `earthkit.utils.units` module provides functions for converting units across
NumPy arrays, xarray DataArrays, and xarray Datasets.

The main entry point is `convert_units`, which automatically dispatches to the
appropriate converter based on the input data type.

In [ ]:
import numpy as np
import xarray as xr

from earthkit.utils.units.convert import convert_units, are_equal

## Checking Unit Equivalence

Use `are_equal` to check whether two unit strings refer to the same physical unit,
even if written differently.

In [ ]:
print(are_equal("m/s", "m s-1"))  # True - same unit, different notation
print(are_equal("m", "km"))        # False - different units

## Converting NumPy Arrays

For plain arrays, both `source_units` and `target_units` must be provided.

In [ ]:
data = np.array([1000.0, 2000.0, 3000.0])
result = convert_units(data, target_units="km", source_units="m")
print(result)  # [1. 2. 3.]

In [ ]:
# Compound units work too
wind_ms = np.array([10.0, 20.0])
wind_kmh = convert_units(wind_ms, target_units="km/h", source_units="m/s")
print(wind_kmh)  # [36. 72.]

## Converting xarray DataArrays

When the input is a DataArray, `source_units` can be omitted. The converter
will read the units from `data.attrs["units"]` automatically. The result's
`units` attribute is updated to the target.

In [ ]:
temperature = xr.DataArray(
    [273.15, 300.0, 310.0],
    dims="time",
    attrs={"units": "K", "long_name": "Temperature"},
)
print("Before:", temperature.values, "|", temperature.attrs["units"])

result = convert_units(temperature, target_units="degC")
print("After: ", result.values, "|", result.attrs["units"])

In [ ]:
# Providing source_units explicitly overrides the attrs
distance = xr.DataArray([1.0, 2.0], attrs={"units": "unknown"})
result = convert_units(distance, target_units="km", source_units="m")
print(result.values)          # [0.001 0.002]
print(result.attrs["units"])  # km

## Converting xarray Datasets

### Single target unit

With a single `target_units` string, all variables with compatible units are
converted. Providing `source_units` as a string acts as a filter—only variables
whose current units match will be converted.

In [ ]:
ds = xr.Dataset({
    "temperature": xr.DataArray([273.15, 300.0], attrs={"units": "K"}),
    "wind_speed":  xr.DataArray([10.0, 20.0], attrs={"units": "m/s"}),
    "pressure":    xr.DataArray([101325.0],       attrs={"units": "Pa"}),
})

# Convert only variables with units "K" -> degC
result = convert_units(ds, target_units="degC", source_units="K")
print("temperature:", result["temperature"].values, result["temperature"].attrs["units"])
print("wind_speed: ", result["wind_speed"].values, result["wind_speed"].attrs["units"])
print("pressure:   ", result["pressure"].values, result["pressure"].attrs["units"])

### Dictionary mapping for per-variable conversion

Pass a dictionary to `target_units` to convert different variables to different
units in a single call. Variables not listed in the dictionary are left unchanged.

In [ ]:
ds = xr.Dataset({
    "temperature": xr.DataArray([273.15, 300.0], attrs={"units": "K"}),
    "wind_speed":  xr.DataArray([10.0, 20.0],    attrs={"units": "m/s"}),
    "pressure":    xr.DataArray([101325.0],       attrs={"units": "Pa"}),
})

result = convert_units(
    ds,
    target_units={
        "temperature": "degC",
        "wind_speed": "km/h",
        # pressure is not listed, so it stays in Pa
    },
)

print("temperature:", result["temperature"].values, result["temperature"].attrs["units"])
print("wind_speed: ", result["wind_speed"].values, result["wind_speed"].attrs["units"])
print("pressure:   ", result["pressure"].values, result["pressure"].attrs["units"])

### Overriding source units with a dictionary

`source_units` can also be a dictionary. This is useful when the metadata is
missing or incorrect for some variables. Variables not listed in the source
dictionary fall back to their `attrs["units"]`.

In [ ]:
ds = xr.Dataset({
    "temperature": xr.DataArray([0.0, 26.85],    attrs={"units": "degC"}),
    "distance":    xr.DataArray([1000.0, 2000.0], attrs={"units": "m"}),
})

result = convert_units(
    ds,
    target_units={"temperature": "degF", "distance": "km"},
    source_units={"temperature": "degC"},  # distance falls back to attrs ("m")
)

print("temperature:", result["temperature"].values, result["temperature"].attrs["units"])
print("distance:   ", result["distance"].values, result["distance"].attrs["units"])

## DataArray with Dictionary Mapping

Dictionary-based target/source units also work with individual DataArrays.
The DataArray's `name` is used as the lookup key.

In [ ]:
da = xr.DataArray(
    [1000.0, 2000.0],
    dims="x",
    name="distance",
    attrs={"units": "m"},
)

result = convert_units(da, target_units={"distance": "km"})
print(result.values)          # [1. 2.]
print(result.attrs["units"])  # km

In [ ]:
# If the DataArray name is not in the dict, no conversion is performed
result = convert_units(da, target_units={"other_var": "km"})
print(result.values)          # [1000. 2000.] - unchanged
print(result.attrs["units"])  # m

## Graceful Handling of Unsupported Conversions

Incompatible or unrecognised units return the data unchanged rather than raising
an error.

In [ ]:
# Incompatible dimensions (length -> temperature)
data = np.array([1.0, 2.0])
result = convert_units(data, target_units="kelvin", source_units="m")
print("Incompatible:", result)  # unchanged

# Unrecognised unit string
result = convert_units(data, target_units="m", source_units="foobar")
print("Unrecognised:", result)  # unchanged